# 📊 03｜圖書館借閱：讀懂三種檔案，再查表對應分類

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/03_library_files/lesson.ipynb)

> 💡 **學習模式說明**：
> 本筆記本專為**完全不寫程式的初學者**設計，全程搭配 **Google Colab 內建的 Gemini AI** 進行協作。
> 你不需要手寫程式碼：先用 Prompt 指揮 AI，再對照下方已附的參考程式，執行並驗收數字與圖表。
> 🆕 **本例只新增一個重點**：資料分散在 **CSV（Big5 編碼）、Excel（兩個工作表）、JSON** 三種檔案；要先正確讀進來，再用「代碼對照表」把借閱紀錄對應到分類名稱。

## 🪜 本例路線：先完成一次，再逐步加深
1. **必做主線（單元一～三）**：讀入三種檔案、把每筆借閱對應到分類名稱，最後用**一張水平長條圖**回答「哪一類書被借最多次？」。單元三後先停下來驗收。
2. **報表應用（單元四）**：匯出分類借閱統計 Excel，並附上每類館藏冊數，避免只看次數下結論。
3. **選做挑戰（單元五～六）**：比較不同身分的借閱量（要注意分母），或觀察借閱天數的分布。

---

## 🧭 資料專案介紹：貫徹「看懂資料與選對圖表」思維 SOP

資料：`data/loans_big5.csv`（借閱紀錄）、`data/books.xlsx`（工作表「書目」與「分類代碼」）、`data/members.json`（會員）。請先閱讀[本例資料卡](./data/DATA_CARD.md)。**借閱檔的一列代表：一次借書紀錄**（600 筆）；書目一列是一本書，會員一列是一位會員。

### 1. 欄位角色貼標籤
- ⚪ **標識 (ID)**：`借閱編號`、`會員編號`、`書號`、`分類號` ➔ 是用來**對照的鑰匙**，只能計數或連結，不能加總平均；`分類號` 必須保持文字（"000" 不能變成 0）。
- 🔵 **條件 (Dimension)**：`分類名稱`、`身分`（類別）；`借書日`、`還書日`（時間）。
- 🟢 **指標 (Metric)**：原檔沒有現成的數值指標；「借閱次數」要用借閱紀錄的**筆數**算出來。

### 2. 五大提問積木與核心問題
- 🎯 **比較（必做）**：哪一個分類的書被借最多次？哪一類最少？
- 👥 **比較＋分母（選做）**：學生、教職員、校友，誰借得比較多？用「總次數」和「每人平均次數」答案一樣嗎？
- 📊 **分布（選做）**：已歸還的書，借閱天數主要集中在幾天？

### 3. 邊界反向檢核防呆（不可觸犯的紅線）
- ⚠️ **先讀對檔案**：借閱檔是 Big5 編碼，要指定 `encoding="cp950"`；Excel 有兩個工作表，預設只會讀到第一個。
- ⚠️ **代碼保持文字**：分類號 "000" 若被讀成數字 0，就對不到「總類」；合併後要檢查有沒有對不到的紀錄。
- ⚠️ **次數不等於受歡迎程度**：各分類的館藏冊數不同，藏書多的分類自然容易被借多；下結論前要附上館藏數。
- ⚠️ **未還不是 0 天**：還書日空白代表尚未歸還，計算借閱天數時要排除並說明筆數，不能當成 0。

### 4. 決定最適圖表
- **8 個分類比借閱次數** ➔ **水平長條圖**（類別較多、名稱較長，依次數排序；數值軸從 0 開始）。
- **3 種身分比總次數與每人平均（選做）** ➔ 兩張並排的**垂直長條圖**，分開呈現兩種不同的指標。
- **借閱天數集中區間（選做）** ➔ **直方圖**（連續數值看次數分布）。

---

## 🤖 Colab Gemini AI 操作方式
1. 每次只複製一個單元的 Prompt，貼到 Colab Gemini 對話框。
2. 比較 AI 的答覆與下方參考程式，執行目前單元的儲存格。
3. 先看懂輸出並回答驗收問題，再進入下一單元。

In [ ]:
# 第 0 步：下載範例資料並安裝繁體中文字型（請先點擊左側 ▶ 執行此儲存格）
import os
from pathlib import Path
from urllib.request import urlretrieve

# 1. 下載資料（已存在就不重複下載；若下載失敗，請手動上傳到 data/）
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
files = {
    'loans_big5.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/03_library_files/data/loans_big5.csv',
    'books.xlsx': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/03_library_files/data/books.xlsx',
    'members.json': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/03_library_files/data/members.json',
}
for name, url in files.items():
    if not (data_dir / name).exists():
        urlretrieve(url, data_dir / name)
print('📂 data/ 內的檔案：', sorted(p.name for p in data_dir.iterdir()))

# 2. 安裝與設定 Colab 繁體中文字型 (思源黑體)
%pip install -q mplfonts
import glob
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)
use_font("Noto Sans CJK SC")

print("✅ 環境初始化完成！資料已就緒，繁體中文字型已設定成功。請開始進行下方單元。")

### 📝 單元一【必做】：讀取三種檔案與欄位角色檢核（貼標籤）

> 🎯 **思維重點**：三種格式要用三種讀法：Big5 的 CSV 指定編碼、Excel 讀兩個工作表、JSON 讀成表格；分類號要保持文字。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請幫我用 pandas 讀取 data/ 中的三個檔案：
1. loans_big5.csv 是 Big5 編碼，請用 encoding="cp950" 讀取。
2. books.xlsx 有「書目」與「分類代碼」兩個工作表，兩個都要讀；「分類號」請指定為文字格式，保留 "000" 這種前導 0。
3. members.json 讀成表格。
4. 每份資料都顯示前 3 列、欄位名稱、列數與缺值數，並說明「一列代表什麼」。
5. 用文字回報：哪些欄位是標識（對照用的代碼）、哪些是條件；本例的數值指標要怎麼得到？
先不要畫圖。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import pandas as pd

loans = pd.read_csv('data/loans_big5.csv', encoding='cp950')           # Big5 編碼
sheets = pd.read_excel('data/books.xlsx', sheet_name=None, dtype={'分類號': str})  # 讀入全部工作表
books, categories = sheets['書目'], sheets['分類代碼']
members = pd.read_json('data/members.json')

for name, table, unit in [('借閱紀錄 loans', loans, '一次借書'), ('書目 books', books, '一本書'),
                          ('分類代碼 categories', categories, '一個分類'), ('會員 members', members, '一位會員')]:
    print(f"\n=== {name}：{len(table)} 列，一列代表「{unit}」 ===")
    display(table.head(3))
    print('缺值數：', table.isna().sum().to_dict())

print('\nExcel 內的工作表：', list(sheets.keys()))
print('分類號範例（應為三位數文字）：', categories['分類號'].tolist())

### 📝 單元二【必做】：邊界檢核：查表對應分類，核對有沒有對不到的紀錄

> 🎯 **思維重點**：先看見「讀錯會怎樣」，再用書號→分類號→分類名稱兩次查表；合併前後筆數必須相同，且沒有對不到的紀錄。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請幫我檢查讀檔設定，並把每筆借閱對應到分類名稱：
1. 示範兩個常見錯誤：用預設編碼讀 loans_big5.csv 會發生什麼事？不指定文字格式時，「分類號」會變成什麼？（只示範，不要用錯誤版本繼續分析）
2. 用「書號」把借閱紀錄對應到書目，再用「分類號」對應到分類名稱；保留所有借閱紀錄（left join）。
3. 列出合併前後的借閱筆數，並統計有幾筆借閱對不到分類名稱。
4. 依分類名稱計算借閱次數（筆數），由多到少排序。
不要修改原始檔案。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
# 示範 1：用預設的 UTF-8 讀 Big5 檔會出錯
try:
    pd.read_csv('data/loans_big5.csv')
except UnicodeDecodeError as e:
    print('❌ 預設編碼讀取失敗：', type(e).__name__)

# 示範 2：不指定文字格式，分類號的前導 0 會消失
wrong = pd.read_excel('data/books.xlsx', sheet_name='分類代碼')
print('❌ 未指定文字格式的分類號：', wrong['分類號'].tolist()[:3], '← "000" 變成 0，就對不到「總類」')

# 正確做法：書號 → 分類號 → 分類名稱，兩次查表，保留所有借閱紀錄
loan_detail = (loans.merge(books[['書號', '書名', '分類號']], on='書號', how='left')
                    .merge(categories, on='分類號', how='left'))
print(f"\n合併前借閱筆數：{len(loans)}；合併後：{len(loan_detail)}")
print('對不到分類名稱的筆數：', loan_detail['分類名稱'].isna().sum())

category_counts = (loan_detail.groupby('分類名稱').size()
                   .sort_values(ascending=False).rename('借閱次數').to_frame())
display(category_counts)
print('各分類次數加總：', category_counts['借閱次數'].sum())

### 📝 單元三【必做】：提問積木【比較】— 哪一類書被借最多次？（水平長條圖）

> 🎯 **思維重點**：問題類別屬於「比較」，但分類有 8 個、名稱較長，依選圖規則改用【水平長條圖】並排序，數值軸從 0 開始。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用 matplotlib 畫出 8 個分類「借閱次數」的水平長條圖：
1. Y 軸為分類名稱，X 軸為借閱次數（次），依次數排序，最多的放在最上面。
2. X 軸數值起點必須從 0 開始。
3. 在每根長條右側標註次數。
4. 借閱次數最多的分類用醒目顏色標示，並寫在標題中。
5. 圖表標題、座標軸與標籤全部使用繁體中文；圖上數字要和上一步的表格一致。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import matplotlib.pyplot as plt

counts = category_counts['借閱次數'].sort_values()          # 由少到多，畫出來最多的在最上面
top_category = counts.idxmax()
colors = ['#e74c3c' if c == top_category else '#3498db' for c in counts.index]

plt.figure(figsize=(8, 5))
bars = plt.barh(counts.index, counts.values, color=colors)
for bar in bars:
    plt.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height() / 2, f'{bar.get_width():.0f}', va='center', fontsize=11)

plt.xlim(0, counts.max() * 1.15)
plt.title(f'各分類借閱次數（最多：{top_category}）', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('借閱次數（次）', fontsize=12)
plt.ylabel('分類', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()

## ✅ 必做主線驗收：現在先停下來

✏️ **先不要看答案！** 對照上面的表格和圖，自己回答下列問題，把你的答案寫在紙上或筆記裡：

- [ ] 能說出三個檔案各自「一列代表什麼」，並指出書號、分類號是對照用的標識。
- [ ] 能展示合併前後借閱筆數不變、沒有對不到分類的紀錄；水平長條圖和次數表的數字一致。
- [ ] 能用一句話解釋：為什麼「被借最多次」的分類，不一定代表「每本書最受歡迎」？
- 💬 **討論題**：如果自然科學類的藏書本來就最多，這個結論要怎麼補充？

<details>
<summary>👉 寫好了嗎？點此展開參考答案核對</summary>

- 借閱紀錄 600 筆、書目 120 本、分類代碼 8 類、會員 80 位；分類號仍是 "000"、"100" 這種三位數文字。
- 對應分類後仍是 600 筆，對不到分類的紀錄為 0 筆。
- 水平長條圖與次數表都顯示**自然科學類**被借最多（124 次）、**哲學類**最少（26 次）。

</details>

若今天繼續做報表，請進行**單元四**；單元五、六是選做挑戰。

---

### 📝 單元四【報表應用】：匯出分類借閱統計，附上館藏冊數

> 🎯 **思維重點**：報表要能防止誤讀：除了借閱次數，也附上每類的**館藏冊數**與「每冊平均借閱次數」。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請整理分類借閱統計並匯出 Excel：
1. 依分類名稱統計：借閱次數、館藏冊數（書目中該分類有幾本書）、每冊平均借閱次數（借閱次數 ÷ 館藏冊數，保留一位小數）。
2. 依借閱次數由多到少排序，存為 output/分類借閱統計.xlsx（若 output 目錄不存在請自動建立）。
3. 匯出後重新讀取，核對借閱次數合計仍為 600。
4. 用一句話說明：借閱次數排名和每冊平均借閱次數排名是否相同。
5. 提供 Colab 下載該 Excel 檔案的程式碼。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
holdings = books.merge(categories, on='分類號', how='left').groupby('分類名稱').size().rename('館藏冊數')
report = category_counts.join(holdings)
report['每冊平均借閱次數'] = (report['借閱次數'] / report['館藏冊數']).round(1)
display(report)

os.makedirs('output', exist_ok=True)
output_path = 'output/分類借閱統計.xlsx'
report.reset_index().to_excel(output_path, index=False)
check = pd.read_excel(output_path)
print('重新讀取後借閱次數合計：', check['借閱次數'].sum())
print('借閱次數最多：', report['借閱次數'].idxmax(), '／ 每冊平均最高：', report['每冊平均借閱次數'].idxmax())

try:
    from google.colab import files
    files.download(output_path)
    print("✅ 已啟動瀏覽器下載 Excel 報表！")
except Exception:
    print(f"✅ Excel 檔案已儲存於：{output_path}")

### 📝 單元五【選做挑戰】：提問積木【比較＋分母】— 哪種身分借得比較多？（兩張垂直長條圖）

> 🎯 **思維重點**：「總次數」受人數影響：學生人數最多，總次數自然最多。改用「每人平均借閱次數」（分母＝該身分會員數）才是公平比較。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用「會員編號」把借閱紀錄對應到 members.json 的身分：
1. 先核對合併後筆數仍為 600、沒有對不到身分的紀錄。
2. 依身分計算：借閱總次數、會員人數（members.json 中該身分的人數）、每人平均借閱次數（總次數 ÷ 會員人數）。
3. 畫兩張並排的垂直長條圖：左邊「借閱總次數」、右邊「每人平均借閱次數」，Y 軸從 0 開始。
4. 用文字說明兩張圖的結論是否不同，以及為什麼要看分母。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
loan_member = loans.merge(members[['會員編號', '身分']], on='會員編號', how='left')
print('合併後筆數：', len(loan_member), '；對不到身分：', loan_member['身分'].isna().sum())

identity = loan_member.groupby('身分').size().rename('借閱總次數').to_frame()
identity['會員人數'] = members.groupby('身分').size()
identity['每人平均借閱次數'] = (identity['借閱總次數'] / identity['會員人數']).round(1)
identity = identity.sort_values('借閱總次數', ascending=False)
display(identity)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, col, unit in [(axes[0], '借閱總次數', '次'), (axes[1], '每人平均借閱次數', '次／人')]:
    ax.bar(identity.index, identity[col], color='#3498db', width=0.5)
    for x, y in zip(identity.index, identity[col]):
        ax.text(x, y * 1.02, f'{y:g}', ha='center')
    ax.set_ylim(0, identity[col].max() * 1.2)
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.set_ylabel(f'{col}（{unit}）')
plt.tight_layout()
plt.show()
print("💡 學生總次數最多，主要是因為學生人數最多；看每人平均，三種身分其實相近。")

### 📝 單元六【選做挑戰】：提問積木【分布】— 借閱天數集中在幾天？（直方圖）

> 🎯 **思維重點**：日期是文字，要先轉成日期才能相減；尚未歸還的紀錄沒有借閱天數，排除並說明筆數，不能當成 0 天。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請分析已歸還書籍的借閱天數：
1. 將借書日、還書日由文字（YYYY/MM/DD）轉成日期格式。
2. 計算借閱天數＝還書日 − 借書日；還書日空白（尚未歸還）的紀錄請排除，並印出排除了幾筆。
3. 畫出借閱天數的直方圖，X 軸為借閱天數（天）、Y 軸為筆數，並畫出平均線與中位數線。
4. 用圖上的數字描述借閱天數主要集中的區間。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
loans_days = loans.copy()
loans_days['借書日'] = pd.to_datetime(loans_days['借書日'], format='%Y/%m/%d')
loans_days['還書日'] = pd.to_datetime(loans_days['還書日'], format='%Y/%m/%d')
returned = loans_days.dropna(subset=['還書日']).copy()
returned['借閱天數'] = (returned['還書日'] - returned['借書日']).dt.days
print(f"已歸還 {len(returned)} 筆；尚未歸還 {loans_days['還書日'].isna().sum()} 筆（不納入天數計算，也不當成 0 天）")

days = returned['借閱天數']
plt.figure(figsize=(8, 4.5))
plt.hist(days, bins=range(0, days.max() + 4, 3), color='#3498db', edgecolor='white')
plt.axvline(days.mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f'平均 {days.mean():.1f} 天')
plt.axvline(days.median(), color='#2c3e50', linestyle=':', linewidth=2, label=f'中位數 {days.median():.0f} 天')
plt.title('已歸還書籍的借閱天數分布', fontsize=13, fontweight='bold')
plt.xlabel('借閱天數（天，每 3 天一組）')
plt.ylabel('筆數')
plt.legend()
plt.show()
print(days.describe().round(1))

---

## 🎯 學習驗收與成果回顧

完成目前選擇的層次後，請檢視你是否能親自驗收：
- [ ] **必做**：正確讀入三種檔案、分類號保留前導 0，合併後 600 筆且無對不到的紀錄；自然科學類借閱最多。
- [ ] **報表應用**：Excel 同時有借閱次數、館藏冊數與每冊平均借閱次數。
- [ ] **選做挑戰**：能用分母說明身分比較的結論；借閱天數排除未還紀錄並說明筆數。
- [ ] **AI 協作**：能說明自己提出的問題、AI 幫忙執行的事，以及自己如何確認結果。